In [ ]:
import librosa
import soundfile as sf
import matplotlib.pyplot as plt
import numpy as np
from scipy import signal
import torchaudio
import os
import sys
sys.path.append(os.path.abspath(".."))
from data_converter import DataConverter


# def stretch_audio(input_file, output_file, target_duration):
#     y, sr = torchaudio.load(input_file)

#     orig_duration = librosa.get_duration(y=y, sr=sr)
#     rate = orig_duration / target_duration

#     y = y.numpy().squeeze()  # Convert to numpy array and remove channel dimension if mono
#     y_out = librosa.effects.time_stretch(y, rate=rate)

#     sf.write(output_file, y_out, sr)
    


# def make_audible_10s(input_file, output_file, factor=10):
#     y, sr = torchaudio.load(input_file)
#     y = y.numpy().squeeze()

#     # Step 1: Slow down by resampling (pitch + duration)
#     new_sr = sr // factor
#     y_slow = librosa.resample(y, orig_sr=sr, target_sr=new_sr)

#     # Step 2: Save at original sr so slowdown is audible
#     y_slow = y_slow / (np.max(np.abs(y_slow)) + 1e-9)

#     # Step 3: Time-stretch to fill 10 seconds
#     current_duration = len(y_slow) / sr
#     stretch_rate = current_duration / 10.0
#     y_stretched = librosa.effects.time_stretch(y_slow, rate=stretch_rate)

#     # Step 4: Pad/crop to exactly 10 seconds
#     target_samples = int(10 * sr)
#     if len(y_stretched) < target_samples:
#         y_stretched = np.pad(y_stretched, (0, target_samples - len(y_stretched)))
#     else:
#         y_stretched = y_stretched[:target_samples]

#     sf.write(output_file, y_stretched, sr)


# def slow_down_audio(input_file, output_file, factor=10):
#     y, sr = librosa.load(input_file, sr=None)

#     new_sr = int(sr / factor)

#     # Write the audio with the LOWER sample rate
#     sf.write(output_file, y, new_sr)

#     print("Original duration:", librosa.get_duration(y=y, sr=sr))
#     print("New duration:", librosa.get_duration(y=y, sr=new_sr))

#     print(f"Old sr: {sr}, New sr: {new_sr}")

# def slow_down_audio_1(input_file, output_file, factor=10):
#     y, sr = librosa.load(input_file, sr=None)

#     dur = librosa.get_duration(y=y, sr=sr)
#     if dur > 1.0:
#         factor /= dur

#     new_sr = int(sr / factor)

#     # Write the audio with the LOWER sample rate
#     sf.write(output_file, y, new_sr)

#     print("Original duration:", librosa.get_duration(y=y, sr=sr))
#     print("New duration:", librosa.get_duration(y=y, sr=new_sr))

#     print(f"Old sr: {sr}, New sr: {new_sr}")

def slow_down_audio_resample(input_file, output_file, target_duration=10.0, target_sr=16000):
    y, sr = librosa.load(input_file, sr=None)
    print(f"Current number of samples: {len(y)}, Current duration: {librosa.get_duration(y=y, sr=sr)}s, current sr: {sr} Hz")
    target_samples = int(target_duration * target_sr)
    
    #pretend original sr is target_samples / target_duration to stretch to target_samples
    y_stretched = librosa.resample(y, orig_sr=len(y)/target_duration, target_sr=target_sr)
    
    sf.write(output_file, y_stretched, target_sr)

    print(f"New duration: {librosa.get_duration(y=y_stretched, sr=target_sr)}s")
    print(f"Final sr: {target_sr} Hz")

def slow_down_audio_resample_padded(input_file, output_file, target_duration=10.0, target_sr=16000):
    y, sr = librosa.load(input_file, sr=None)
    
    pad_duration = 0.5
    core_duration = target_duration - (2 * pad_duration)
    
    y_stretched = librosa.resample(y, orig_sr=len(y)/core_duration, target_sr=target_sr)
    
    pad_samples = int(pad_duration * target_sr) 
    silence = np.zeros(pad_samples)
    y_final = np.concatenate((silence, y_stretched, silence))
    
    sf.write(output_file, y_final, target_sr)

    print(f"Core stretched audio: {librosa.get_duration(y=y_stretched, sr=target_sr):.2f}s")
    print(f"Total padded duration: {librosa.get_duration(y=y_final, sr=target_sr):.2f}s")
    print(f"Final sr: {target_sr} Hz")


In [ ]:
base_folder = '/scratch/local/hdd/hani/dolphins/click_trains/'
random_folder = np.random.choice(os.listdir(base_folder))
print("Selected folder:", random_folder)
input_file = os.path.join(base_folder, random_folder, 'PulseTrain.wav')
txt_file = os.path.join(base_folder, random_folder, 'PulseParameters.txt')
with open(txt_file, 'r') as file:
        file_content = file.read()
        print(file_content)

output_file = 'PulseTrain_altered.wav'
target_duration = 10.0  # seconds
#stretch_audio(input_file, output_file, target_duration)
#slow_down_audio_1(input_file, output_file)
#stretch_audio(output_file, output_file, target_duration)
#make_audible_10s(input_file, output_file, factor=20)
slow_down_audio_resample(input_file, output_file, target_duration=10.0, target_sr=16000)
# slow_down_audio_resample_padded(input_file, output_file, target_duration=10.0, target_sr=16000)

In [ ]:
#display spectrogram
import shutil

file_path = output_file

y, sr = torchaudio.load(os.path.expanduser(file_path))
print(f"Length of audio: {y.shape[1] / sr:.2f} seconds")
y = y.numpy()

waveform = y.mean(axis=0)
_, _, spectrogram = signal.spectrogram(waveform, sr, nperseg=512, noverlap=256)
spectrogram = np.log(spectrogram + 1e-7)

converter = DataConverter()
spectrogram = converter.apply_histogram_equalisation(spectrogram, method="global")

mean = np.mean(spectrogram)
std = np.std(spectrogram)
spectrogram = np.divide(spectrogram - mean, std + 1e-9)
print(spectrogram.shape)

if spectrogram.ndim == 3:
    spectrogram = spectrogram.mean(axis=0)

plt.figure(figsize=(10, 10))
plt.imshow(spectrogram, origin='lower', aspect='equal', cmap="magma")
plt.axis('off')
plt.show()

#copy file to home directory
save_loc = '/users/hani/AudioCounting/test_dolphin.wav'
shutil.copy(output_file, save_loc)

In [ ]:
from tqdm.notebook import tqdm
import os

base_folder = '/scratch/local/hdd/hani/dolphins/click_trains/'
output_folder = '/scratch/local/hdd/hani/dolphins/test_padded/'
padded = True

os.makedirs(output_folder, exist_ok=True)

for f in os.listdir(output_folder):
    os.remove(os.path.join(output_folder, f))

folders = [
    f for f in os.listdir(base_folder)
    if os.path.isdir(os.path.join(base_folder, f))
]

for folder in tqdm(folders, total=len(folders), desc="Processing"):
    input_file = os.path.join(base_folder, folder, 'PulseTrain.wav')
    output_file = os.path.join(output_folder, f'{folder}.wav')

    if not os.path.exists(input_file):
        print(f"Skipping {folder}: no PulseTrain.wav")
        continue

    if padded:
        slow_down_audio_resample_padded(input_file, output_file,
                                        target_duration=10.0, target_sr=16000)
    else:
        slow_down_audio_resample(input_file, output_file,
                                 target_duration=10.0, target_sr=16000)


In [ ]:
#Create csv of pulse counts
import csv

base_folder = '/scratch/local/hdd/hani/dolphins/click_trains/'
output_csv = '/users/hani/AudioCounting/preprocessing/dolphins/dolphins.csv'

if not os.path.exists(base_folder):
    raise FileNotFoundError(f"Base folder '{base_folder}' does not exist.")

rows = []

for folder in sorted(os.listdir(base_folder)):
    folder_path = os.path.join(base_folder, folder)
    params_file = os.path.join(folder_path, 'PulseParameters.txt')
    wav_file = f"{folder}.wav"

    if not os.path.isdir(folder_path):
        continue
    if not os.path.exists(params_file):
        continue

    with open(params_file, 'r') as f:
        lines = f.readlines()

    data_lines = [ln for ln in lines[1:] if ln.strip()]

    pulse_count = len(data_lines)

    rows.append([wav_file, pulse_count])

num_under_9 = sum(1 for _, count in rows if count < 9)
print(f"Number of samples with fewer than 9 pulses: {num_under_9}")

with open(output_csv, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(["location", "repetitions"])
    writer.writerows(rows)

print(f"Saved CSV to {output_csv}")
